In [0]:
fact_admissions = spark.sql(f"select * from regis_healthcare.silver.admissions;")
fact_admissions.createOrReplaceTempView("admissions")

In [0]:
# Fact_Admissions --> Source: admissions

# | Foreign Keys       |
# | ------------------ |
# | admission_key      |
# | resident_key       |
# | facility_key       |
# | admission_date_key |

#--------------------------

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_admissions = fact_admissions.withColumn("admission_date_key", date_format(col("admission_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_admissions = fact_admissions.withColumn("admission_date_key", col("admission_date_key").cast("int"))
#--------------------------
from pyspark.sql.functions import col, regexp_replace
fact_admissions = fact_admissions.withColumn(
    "admission_key",
    regexp_replace(col("admission_id"), "^ADM", "").cast("int")
)
fact_admissions = fact_admissions.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_admissions = fact_admissions.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
# display(df_facilities)
fact_admissions = fact_admissions.select(
 "admission_key",   
 "resident_key",       
 "facility_key",       
 "admission_date_key"
)
display(fact_admissions)



#### cataloge 

In [0]:
fact_admissions.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_admissions")

In [0]:
fact_admissions.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_admissions")
print(fact_admissions.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_admissions")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_admissions")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.admission_key = source.admission_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_admissions;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_admissions;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_admissions.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_admissions")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_admissions"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_admissions

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.admission_key = source.admission_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
